In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [2]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [3]:
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [4]:
df_Ca= df[df['State'] == 'CA']

In [5]:
df_Ca.shape

(1741433, 46)

In [6]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [7]:

df_Ca=df_Ca[columns]

In [8]:
df_Ca.columns

Index(['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Temperature(F)',
       'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Crossing',
       'Junction', 'Railway', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset'],
      dtype='object')

In [9]:
df_Ca.notnull().sum()

Severity             1741433
Start_Time           1741433
Start_Lat            1741433
Start_Lng            1741433
Temperature(F)       1695464
Wind_Chill(F)        1230468
Humidity(%)          1693092
Pressure(in)         1704307
Visibility(mi)       1701308
Wind_Speed(mph)      1578542
Precipitation(in)    1175229
Weather_Condition    1701655
Crossing             1741433
Junction             1741433
Railway              1741433
Stop                 1741433
Traffic_Signal       1741433
Sunrise_Sunset       1740090
dtype: int64

In [10]:
df_Ca.drop(columns=['Wind_Chill(F)', 'Precipitation(in)'], inplace=True)

In [11]:
df_Ca.fillna({
    'Temperature(F)': df_Ca['Temperature(F)'].mean(),
    'Humidity(%)': df_Ca['Humidity(%)'].mean(),
    'Pressure(in)': df_Ca['Pressure(in)'].mean(),
    'Visibility(mi)': df_Ca['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_Ca['Wind_Speed(mph)'].mean()
},inplace=True)

In [12]:
df_Ca.fillna({
    'Weather_Condition': df_Ca['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_Ca['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [13]:
df_Ca['Start_Time'] = pd.to_datetime(df_Ca['Start_Time'], errors='coerce')
df_Ca['hour']= df_Ca['Start_Time'].dt.hour
df_Ca['day']= df_Ca['Start_Time'].dt.dayofweek
df_Ca['month']= df_Ca['Start_Time'].dt.month

In [14]:
df_Ca.sample(12)
df_Ca.notnull().sum()
df_Ca.fillna({
    'hour': df_Ca['hour'].mode()[0],
    'day': df_Ca['day'].mode()[0],
    'month': df_Ca['month'].mode()[0]
}, inplace=True)

In [15]:
df_Ca.notnull().sum()
df_Ca.drop(columns= ['Start_Time'], inplace=True)

In [16]:

df_Ca.notnull().sum()

Severity             1741433
Start_Lat            1741433
Start_Lng            1741433
Temperature(F)       1741433
Humidity(%)          1741433
Pressure(in)         1741433
Visibility(mi)       1741433
Wind_Speed(mph)      1741433
Weather_Condition    1741433
Crossing             1741433
Junction             1741433
Railway              1741433
Stop                 1741433
Traffic_Signal       1741433
Sunrise_Sunset       1741433
hour                 1741433
day                  1741433
month                1741433
dtype: int64

In [17]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [26]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [27]:
x= df_Ca.drop(columns=['Severity'])
y= df_Ca['Severity']


In [28]:
y1=df_Ca['Severity']-1

In [29]:
y1 = (df_Ca['Severity'] >= 3).astype(int)

In [31]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y1, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.32).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8138680352452451

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.65      0.78    176081
           1       0.37      0.93      0.53     38643

    accuracy                           0.70    214724
   macro avg       0.67      0.79      0.66    214724
weighted avg       0.87      0.70      0.74    214724


Confusion Matrix:
 [[114756  61325]
 [  2713  35930]]


In [32]:
import joblib
joblib.dump(model2,"california_model1.pkl")

['california_model1.pkl']

In [23]:
import numpy as np

df_Ca = df_Ca.copy()

# remove impossible coordinates
df_Ca = df_Ca[
    (df_Ca['Start_Lat'].between(-90, 90)) &
    (df_Ca['Start_Lng'].between(-180, 180))
]

# remove impossible weather values
df_Ca = df_Ca[df_Ca['Temperature(F)'] > -100]   # sanity check
df_Ca = df_Ca[df_Ca['Wind_Speed(mph)'] >= 0]
df_Ca = df_Ca[df_Ca['Pressure(in)'] > 0]
df_Ca = df_Ca[df_Ca['Humidity(%)'].between(0, 100)]
df_Ca = df_Ca[df_Ca['Visibility(mi)'] >= 0]

In [24]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return df[(df[col] >= lower) & (df[col] <= upper)]

numeric_cols = [
    'Temperature(F)',
    'Wind_Speed(mph)',
    'Pressure(in)',
    'Visibility(mi)',
    'Humidity(%)'
]

for col in numeric_cols:
    df_Ca = iqr_filter(df_Ca, col)

In [25]:
lat_min, lat_max = df_Ca['Start_Lat'].quantile([0.01, 0.99])
lng_min, lng_max = df_Ca['Start_Lng'].quantile([0.01, 0.99])

df_Ca = df_Ca[
    (df_Ca['Start_Lat'].between(lat_min, lat_max)) &
    (df_Ca['Start_Lng'].between(lng_min, lng_max))
]